# Miniproyecto 3 – Redes generativas
## Obtención de una imagen completa a partir de un bosquejo (Sketch → Foto) con **Pix2Pix**

**Curso:** Redes Neuronales y Deep Learning – Universidad Autónoma de Occidente
**Integrantes:** _(completar)_
**Framework:** PyTorch (el mismo código se puede portar a TensorFlow-Keras)

**Idea del proyecto.** Entrenamos una red generativa condicional (cGAN, arquitectura *Pix2Pix*) que recibe un **bosquejo de bordes** y genera la **fotografía completa** correspondiente.
El dataset es **personalizado**: se construye a partir de fotos propias; el bosquejo de cada foto se genera automáticamente con detección de bordes (Canny), de modo que cada foto produce un par supervisado *(bosquejo, foto)*.

**Contenido del notebook**
1. Configuración
2. Dataset personalizado (construcción y partición train / val / test)
3. Arquitectura: generador U-Net + discriminador PatchGAN
4. Función de pérdida y entrenamiento
5. Validación: curvas, métricas (L1, PSNR, SSIM), comparación visual y prueba con dibujos a mano
6. Guardado del modelo y conclusiones

## 1. Configuración

> En **Google Colab** activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
> Colab ya trae PyTorch, OpenCV, scikit-image y matplotlib. En un equipo local: `pip install torch torchvision opencv-python scikit-image matplotlib tqdm`.

In [ ]:
import math, random, time, zipfile
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from tqdm.auto import tqdm

# ---------------- Hiperparámetros (se ajustan aquí) ----------------
SEED         = 42
IMG_SIZE     = 256     # potencia de 2 (64, 128, 256). Con 256 el cuello de botella llega a 1x1
BATCH_SIZE   = 4
EPOCHS       = 100
LR           = 2e-4
BETA1        = 0.5
LAMBDA_L1    = 100     # peso de la pérdida L1 (valor usado en el paper de Pix2Pix)
NGF          = 64      # ancho base del generador
NDF          = 64      # ancho base del discriminador
DATA_DIR     = Path("datos/fotos")   # carpeta con TUS fotos
OUT_DIR      = Path("resultados")
SAMPLE_EVERY = 10      # cada cuántas épocas guardar/mostrar ejemplos

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
(OUT_DIR / "samples").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
print("Dispositivo:", device)

## 2. Dataset personalizado

**Dataset usado por el grupo:** fotografías de paisajes (montañas, lagos, bosques) de la carpeta compartida `Dataset_CycleGAN_Pintura_Fotografia` en Google Drive (≈170 imágenes de las carpetas `trainB/valB/testB_Fotografias`). Las pinturas (carpetas *A*) no se usan, porque Pix2Pix necesita pares exactos entrada→salida y aquí los pares (bosquejo, foto) se generan a partir de cada foto. La celda de abajo monta Drive y copia las fotos automáticamente.

**Cómo se construye**
1. Se reúne una colección de fotos de **una misma temática** (por ejemplo: zapatillas, plantas, fachadas de casas, platos de comida, paisajes de tu ciudad, objetos de tu escritorio…). Entre **200 y 1000 imágenes** es un rango razonable; más homogéneas = mejores resultados.
   Una forma rápida de conseguirlas es **grabar un video corto** del objeto/escena y extraer fotogramas (función `extract_frames` de abajo).
2. Para cada foto se calcula el **bosquejo** con detección de bordes (Canny automático) → trazo negro sobre fondo blanco.
3. Cada par *(bosquejo, foto)* es un ejemplo de entrenamiento: entrada **x** = bosquejo, objetivo **y** = foto real.

**Preprocesamiento:** recorte cuadrado central → redimensionado → imágenes normalizadas a **[-1, 1]** (porque el generador termina en `tanh`).
**Aumento de datos (solo train):** *random crop* + *flip* horizontal. El bosquejo se calcula *después* del aumento, así que siempre es coherente con la foto.

**Partición:** 80 % entrenamiento, 10 % validación, 10 % prueba (se hace por archivo con semilla fija → reproducible).

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT)

# ---- Solo en Colab: cargar el dataset desde Google Drive ----
# Dataset del grupo: carpeta "Dataset_CycleGAN_Pintura_Fotografia" (compartida en Drive).
# Para Pix2Pix usamos SOLO las fotografías (carpetas *B_Fotografias); el bosquejo se genera con Canny.
# Antes de correr: en Drive → "Compartidos conmigo" → clic derecho en la carpeta →
# "Organizar" → "Agregar acceso directo" → "Mi unidad".
DRIVE_DATASET = "Dataset_CycleGAN_Pintura_Fotografia"
PHOTO_SUBDIRS = ["trainB_Fotografias", "valB_Fotografias", "testB_Fotografias"]

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not list_images(DATA_DIR):
    import shutil
    drive.mount("/content/drive")
    cand = [Path("/content/drive/MyDrive") / DRIVE_DATASET] + \
           sorted(Path("/content/drive/MyDrive").glob(f"**/{DRIVE_DATASET}"))
    src = next((c for c in cand if c.exists()), None)
    if src is not None:
        for sub in PHOTO_SUBDIRS:
            for f in list_images(src / sub):
                shutil.copy(f, DATA_DIR / f.name)
        print(f"Fotos copiadas desde {src}: {len(list_images(DATA_DIR))}")
    else:
        # Plan B: subir un .zip con las fotos
        print(f"No se encontró '{DRIVE_DATASET}' en Mi unidad. Sube un .zip con las fotos:")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(DATA_DIR)
            else:
                Path(name).replace(DATA_DIR / name)

# ---- Opcional: extraer fotogramas de un video para armar el dataset ----
def extract_frames(video_path, out_dir=DATA_DIR, every_n=15):
    # Guarda 1 de cada `every_n` fotogramas del video como imagen .jpg
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cap, i, saved = cv2.VideoCapture(str(video_path)), 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % every_n == 0:
            cv2.imwrite(str(out_dir / f"{Path(video_path).stem}_{i:06d}.jpg"), frame)
            saved += 1
        i += 1
    cap.release()
    print(f"{saved} fotogramas guardados en {out_dir}")
# extract_frames("mi_video.mp4", every_n=15)

all_files = list_images(DATA_DIR)
print("Imágenes encontradas:", len(all_files))
assert len(all_files) >= 10, f"Pon tus fotos en {DATA_DIR} (mínimo 10; lo recomendable es 200 o más)."
if len(all_files) < 200:
    print("Aviso: con menos de ~200 imágenes el modelo funciona, pero la calidad será limitada.")

In [ ]:
# ------------------------- Generación del bosquejo -------------------------
def auto_canny(gray, sigma=0.33):
    # Canny con umbrales calculados a partir de la mediana de la imagen (robusto a la iluminación)
    v = np.median(gray)
    lo, hi = int(max(0, (1.0 - sigma) * v)), int(min(255, (1.0 + sigma) * v))
    return cv2.Canny(gray, lo, hi)

def make_sketch(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)                 # quita ruido antes de buscar bordes
    edges = auto_canny(gray)
    edges = cv2.dilate(edges, np.ones((2, 2), np.uint8))     # trazo un poco más grueso
    return 255 - edges                                       # fondo blanco, trazo negro

def center_square(img):
    h, w = img.shape[:2]
    s = min(h, w)
    y, x = (h - s) // 2, (w - s) // 2
    return img[y:y + s, x:x + s]

def denorm(t):
    # de [-1, 1] a [0, 1] para visualizar
    return (t * 0.5 + 0.5).clamp(0, 1)

# ------------------------------ Dataset ------------------------------------
class SketchToPhotoDataset(Dataset):
    def __init__(self, files, img_size=256, train=True):
        self.files, self.img_size, self.train = list(files), img_size, train

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        bgr = cv2.imread(str(self.files[i]))
        assert bgr is not None, f"No se pudo leer {self.files[i]}"
        img = center_square(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        if self.train:                                        # aumento de datos
            big = int(self.img_size * 1.12)
            img = cv2.resize(img, (big, big), interpolation=cv2.INTER_AREA)
            y = random.randint(0, big - self.img_size)
            x = random.randint(0, big - self.img_size)
            img = img[y:y + self.img_size, x:x + self.img_size]
            if random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1])
        else:
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        sketch = make_sketch(img)
        photo_t  = torch.from_numpy(img).permute(2, 0, 1).float() / 127.5 - 1      # 3 x H x W en [-1,1]
        sketch_t = torch.from_numpy(sketch)[None].float() / 127.5 - 1              # 1 x H x W en [-1,1]
        return sketch_t, photo_t

# ------------------------- Partición train / val / test --------------------
files = list(all_files)
random.Random(SEED).shuffle(files)
n = len(files)
n_test = max(1, int(0.1 * n)); n_val = max(1, int(0.1 * n))
test_files, val_files, train_files = files[:n_test], files[n_test:n_test + n_val], files[n_test + n_val:]
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

train_ds = SketchToPhotoDataset(train_files, IMG_SIZE, train=True)
val_ds   = SketchToPhotoDataset(val_files,   IMG_SIZE, train=False)
test_ds  = SketchToPhotoDataset(test_files,  IMG_SIZE, train=False)

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          pin_memory=pin, drop_last=len(train_ds) >= BATCH_SIZE)
val_loader   = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ------------------------- Ejemplos del dataset ----------------------------
fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    s, p = train_ds[j % len(train_ds)]
    axs[0, j].imshow(denorm(s)[0], cmap="gray"); axs[0, j].axis("off"); axs[0, j].set_title("Boceto (entrada x)")
    axs[1, j].imshow(denorm(p).permute(1, 2, 0));  axs[1, j].axis("off"); axs[1, j].set_title("Foto (objetivo y)")
plt.tight_layout(); plt.show()

## 3. Arquitectura

**Pix2Pix** es una GAN *condicional* y *supervisada*: el generador no parte de ruido aleatorio sino del bosquejo **x**, y el discriminador siempre ve el **par** (x, imagen).

### Generador G – U-Net
* **Encoder** (camino de bajada): bloques `Conv 4×4, stride 2 → BatchNorm → LeakyReLU(0.2)`. Cada bloque reduce a la mitad el tamaño espacial y aumenta los canales (64 → 128 → 256 → 512 …). Con 256×256 son 8 bloques y se llega a un mapa de 1×1 (*cuello de botella*).
* **Decoder** (camino de subida): bloques `ConvTranspose 4×4, stride 2 → BatchNorm → (Dropout 50 %) → ReLU`, que van recuperando el tamaño original.
* **Skip connections:** la salida de cada bloque del encoder se **concatena** con el bloque equivalente del decoder. Así los bordes y detalles espaciales del bosquejo no se pierden en el cuello de botella.
* **Dropout** en las 3 primeras capas del decoder: es la única fuente de estocasticidad (reemplaza al vector de ruido *z* de las GAN clásicas).
* Salida: `ConvTranspose → tanh` → imagen RGB en [-1, 1].

### Discriminador D – PatchGAN (70×70)
Recibe la concatenación **(boceto, imagen)** y entrega un mapa de logits de 30×30 (para 256×256). Cada valor decide si un *parche* de 70×70 píxeles es real o falso. Esto obliga al generador a producir **texturas y detalle local** realistas y es una red totalmente convolucional (pocos parámetros, funciona con cualquier tamaño).

### Función de pérdida
$$G^* = \arg\min_G \max_D \; \mathcal{L}_{cGAN}(G,D) + \lambda\,\mathcal{L}_{L1}(G)$$

* $\mathcal{L}_{cGAN}$: entropía cruzada binaria; D aprende a distinguir (x, y real) de (x, G(x)); G intenta engañarlo.
* $\mathcal{L}_{L1} = \mathbb{E}\,\|y - G(x)\|_1$: mantiene la salida cerca de la foto real (color y estructura global); la GAN aporta nitidez. Con $\lambda = 100$.

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=not norm)]
        if norm: layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False), nn.BatchNorm2d(out_ch)]
        if dropout: layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=3, img_size=256, ngf=64):
        super().__init__()
        n_down = int(math.log2(img_size))                      # 256 -> 8 bloques (llega a 1x1)
        ch = [min(ngf * 2 ** i, 512) for i in range(n_down)]   # 64,128,256,512,512,...
        # Encoder: sin normalización en la primera capa ni en el cuello de botella
        self.down, prev = nn.ModuleList(), in_ch
        for i, c in enumerate(ch):
            self.down.append(DownBlock(prev, c, norm=(0 < i < n_down - 1)))
            prev = c
        # Decoder: la entrada de cada bloque es la salida anterior (+ skip concatenado)
        self.up, dec_in = nn.ModuleList(), ch[-1]
        for k in range(n_down - 1):
            skip_ch = ch[n_down - 2 - k]
            self.up.append(UpBlock(dec_in, skip_ch, dropout=(k < 3)))
            dec_in = skip_ch * 2                               # canales tras concatenar el skip
        self.final = nn.Sequential(nn.ConvTranspose2d(dec_in, out_ch, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        feats = []
        for d in self.down:
            x = d(x); feats.append(x)
        x = feats[-1]                                          # cuello de botella
        for up, skip in zip(self.up, reversed(feats[:-1])):
            x = torch.cat([up(x), skip], dim=1)                # skip connection
        return self.final(x)

class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=1 + 3, ndf=64):
        super().__init__()
        def conv(i, o, stride, norm=True):
            layers = [nn.Conv2d(i, o, 4, stride, 1, bias=not norm)]
            if norm: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *conv(in_ch,   ndf,     2, norm=False),
            *conv(ndf,     ndf * 2, 2),
            *conv(ndf * 2, ndf * 4, 2),
            *conv(ndf * 4, ndf * 8, 1),
            nn.Conv2d(ndf * 8, 1, 4, 1, 1),                    # mapa de logits (un valor por parche)
        )
    def forward(self, sketch, img):
        return self.model(torch.cat([sketch, img], dim=1))

def init_weights(m):
    name = m.__class__.__name__
    if "Conv" in name:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

G = UNetGenerator(1, 3, IMG_SIZE, NGF).to(device); G.apply(init_weights)
D = PatchDiscriminator(1 + 3, NDF).to(device);     D.apply(init_weights)

count = lambda m: sum(p.numel() for p in m.parameters())
print(f"Parámetros  G: {count(G)/1e6:.1f} M   D: {count(D)/1e6:.2f} M")

# Comprobación de formas
with torch.no_grad():
    s, p = next(iter(val_loader))
    s, p = s.to(device), p.to(device)
    print("Boceto:", tuple(s.shape), "-> G(x):", tuple(G(s).shape), "-> D(x, y):", tuple(D(s, p).shape))

## 4. Entrenamiento

En cada iteración:
1. **Paso del discriminador:** `loss_D = ½ [ BCE(D(x,y), 1) + BCE(D(x,G(x)), 0) ]`
2. **Paso del generador:** `loss_G = BCE(D(x,G(x)), 1) + λ · L1(G(x), y)`

Se usa Adam (`lr = 2e-4`, `β₁ = 0.5`) y un *scheduler* que mantiene el *learning rate* constante la primera mitad y lo reduce linealmente a 0 en la segunda. Cada `SAMPLE_EVERY` épocas se guardan ejemplos (fila 1: boceto, fila 2: generada, fila 3: real) y se conserva el mejor checkpoint según la pérdida L1 de validación.

In [ ]:
bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))
half = EPOCHS // 2
lr_rule = lambda e: 1.0 - max(0, e - half) / float(EPOCHS - half + 1)
sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_rule)
sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_rule)

fixed_sketch, fixed_real = next(iter(DataLoader(val_ds, batch_size=min(4, len(val_ds)), shuffle=False)))

@torch.no_grad()
def save_samples(epoch, show=False):
    G.eval()
    fake = G(fixed_sketch.to(device)).cpu()
    rows = torch.cat([denorm(fixed_sketch).repeat(1, 3, 1, 1), denorm(fake), denorm(fixed_real)], dim=0)
    path = OUT_DIR / "samples" / f"epoch_{epoch:03d}.png"
    save_image(rows, path, nrow=fixed_sketch.size(0), padding=2)
    if show:
        plt.figure(figsize=(12, 9)); plt.imshow(plt.imread(path)); plt.axis("off")
        plt.title(f"Época {epoch}  |  fila 1: boceto · fila 2: generada · fila 3: real"); plt.show()
    G.train()

@torch.no_grad()
def val_l1(loader):
    G.eval()
    tot, cnt = 0.0, 0
    for s, p in loader:
        s, p = s.to(device), p.to(device)
        tot += l1(G(s), p).item() * s.size(0); cnt += s.size(0)
    G.train()
    return tot / cnt

In [ ]:
history = {"loss_D": [], "loss_G_gan": [], "loss_G_l1": [], "val_l1": []}
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    G.train(); D.train()
    t0, acc = time.time(), np.zeros(3)
    for sketch, real in tqdm(train_loader, desc=f"Época {epoch}/{EPOCHS}", leave=False):
        sketch, real = sketch.to(device), real.to(device)
        fake = G(sketch)

        # ---- 1) Discriminador ----
        opt_D.zero_grad()
        pred_real = D(sketch, real)
        pred_fake = D(sketch, fake.detach())
        loss_D = 0.5 * (bce(pred_real, torch.ones_like(pred_real)) +
                        bce(pred_fake, torch.zeros_like(pred_fake)))
        loss_D.backward(); opt_D.step()

        # ---- 2) Generador ----
        opt_G.zero_grad()
        pred_fake = D(sketch, fake)
        loss_gan = bce(pred_fake, torch.ones_like(pred_fake))
        loss_l1  = l1(fake, real)
        loss_G   = loss_gan + LAMBDA_L1 * loss_l1
        loss_G.backward(); opt_G.step()

        acc += np.array([loss_D.item(), loss_gan.item(), loss_l1.item()])

    sched_G.step(); sched_D.step()
    acc /= len(train_loader)
    v = val_l1(val_loader)
    for k, x in zip(history.keys(), [*acc, v]):
        history[k].append(float(x))
    print(f"Época {epoch:3d}/{EPOCHS} | D {acc[0]:.3f} | G_gan {acc[1]:.3f} | G_L1 {acc[2]:.3f} | "
          f"val_L1 {v:.3f} | {time.time() - t0:.0f}s")

    if v < best_val:
        best_val = v
        torch.save(G.state_dict(), OUT_DIR / "checkpoints" / "G_best.pth")
    if epoch == 1 or epoch % SAMPLE_EVERY == 0 or epoch == EPOCHS:
        save_samples(epoch, show=True)
        torch.save({"G": G.state_dict(), "D": D.state_dict(), "epoch": epoch},
                   OUT_DIR / "checkpoints" / "last.pth")

## 5. Validación del modelo

Una GAN no tiene una única métrica de "precisión", así que se valida con **varias evidencias**:

1. **Curvas de pérdida:** estabilidad del entrenamiento (D y G deben mantenerse en equilibrio; L1 debe bajar).
2. **Métricas cuantitativas** sobre el conjunto de **prueba** (imágenes que el modelo nunca vio): L1, **PSNR** y **SSIM** entre la imagen generada y la foto real. _(Nota: la L1 de las curvas está en escala [-1, 1] y la de la tabla de test en [0, 1], por eso esta última es la mitad.)_
3. **Comparación visual** boceto · generada · real.
4. **Prueba con bosquejos dibujados a mano** (generalización fuera del dataset).
5. *(Opcional)* **FID**, una métrica estándar para calidad/diversidad en GAN.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].plot(history["loss_D"]);      axs[0].set_title("Pérdida del discriminador (train)")
axs[1].plot(history["loss_G_gan"]);  axs[1].set_title("Pérdida adversaria del generador (train)")
axs[2].plot(history["loss_G_l1"], label="train"); axs[2].plot(history["val_l1"], label="validación")
axs[2].set_title("Pérdida L1 (escala [-1, 1])"); axs[2].legend()
for a in axs: a.set_xlabel("Época"); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "curvas_perdida.png", dpi=150); plt.show()

In [ ]:
# Usamos el mejor checkpoint según L1 de validación
G.load_state_dict(torch.load(OUT_DIR / "checkpoints" / "G_best.pth", map_location=device))
G.eval()

@torch.no_grad()
def evaluate(loader):
    l1s, psnrs, ssims = [], [], []
    for s, p in loader:
        fake = G(s.to(device)).cpu()
        for f, r in zip(fake, p):
            f = denorm(f).permute(1, 2, 0).numpy(); r = denorm(r).permute(1, 2, 0).numpy()
            l1s.append(np.abs(f - r).mean())
            psnrs.append(psnr_fn(r, f, data_range=1.0))
            ssims.append(ssim_fn(r, f, channel_axis=2, data_range=1.0))
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in
            {"L1 (escala 0-1)": l1s, "PSNR (dB)": psnrs, "SSIM": ssims}.items()}

res = evaluate(test_loader)
print(f"Resultados en TEST ({len(test_ds)} imágenes):")
for k, (m, sd) in res.items():
    print(f"  {k:18s}: {m:.4f} ± {sd:.4f}")

In [ ]:
@torch.no_grad()
def show_results(ds, n=6, title=""):
    n = min(n, len(ds))
    fig, axs = plt.subplots(n, 3, figsize=(9, 3 * n))
    axs = np.atleast_2d(axs)
    for i in range(n):
        s, p = ds[i]
        f = G(s[None].to(device)).cpu()[0]
        for ax, im, t in zip(axs[i], [denorm(s).repeat(3, 1, 1), denorm(f), denorm(p)],
                             ["Boceto (entrada)", "Generada por G", "Foto real"]):
            ax.imshow(im.permute(1, 2, 0)); ax.axis("off")
            if i == 0: ax.set_title(t)
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(OUT_DIR / f"comparacion_{title or 'test'}.png", dpi=150); plt.show()

show_results(test_ds, n=6, title="test")

In [ ]:
# ---- Prueba con bosquejos dibujados a mano ----
# Pon tus dibujos (trazo oscuro sobre fondo claro) en datos/bocetos/ . Pueden ser fotos de papel o hechos en Paint.
# Ojo: el modelo se entrenó con bordes tipo Canny, así que los dibujos muy distintos a ese estilo
# pueden dar resultados peores (es un hallazgo válido para discutir en el informe).
@torch.no_grad()
def sketch_file_to_photo(path):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(center_square(g), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    if g.mean() < 127:                       # asegurar trazo oscuro sobre fondo claro
        g = 255 - g
    x = torch.from_numpy(g)[None, None].float() / 127.5 - 1
    G.eval()
    y = G(x.to(device)).cpu()[0]
    return g, denorm(y).permute(1, 2, 0).numpy()

drawn = list_images("datos/bocetos") if Path("datos/bocetos").exists() else []
if not drawn:
    print("No hay dibujos en datos/bocetos/ todavía (carpeta opcional).")
else:
    fig, axs = plt.subplots(len(drawn), 2, figsize=(7, 3.5 * len(drawn)))
    axs = np.atleast_2d(axs)
    for i, pth in enumerate(drawn):
        sk, out = sketch_file_to_photo(pth)
        axs[i, 0].imshow(sk, cmap="gray"); axs[i, 0].axis("off"); axs[i, 0].set_title("Dibujo")
        axs[i, 1].imshow(out);             axs[i, 1].axis("off"); axs[i, 1].set_title("Generada")
    plt.tight_layout(); plt.show()

In [ ]:
# ---- (Opcional) FID: pip install torchmetrics torch-fidelity ----
# Con pocas imágenes de prueba el FID es poco confiable; úsalo solo como referencia.
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    with torch.no_grad():
        for s, p in test_loader:
            f = G(s.to(device))
            fid.update(denorm(p).to(device), real=True)
            fid.update(denorm(f), real=False)
    print("FID (menor es mejor):", float(fid.compute()))
except Exception as e:
    print("FID omitido:", type(e).__name__, "-", e)

In [ ]:
torch.save(G.state_dict(), OUT_DIR / "generador_pix2pix_sketch2photo.pth")
print("Modelo guardado en", OUT_DIR / "generador_pix2pix_sketch2photo.pth")

# Para volver a usarlo:
#   G = UNetGenerator(1, 3, IMG_SIZE, NGF).to(device)
#   G.load_state_dict(torch.load("resultados/generador_pix2pix_sketch2photo.pth", map_location=device)); G.eval()

## 6. Análisis y conclusiones _(completar con TUS resultados)_

**Qué debe quedar escrito aquí / en el informe**
* **Dataset:** temática elegida, cuántas imágenes, de dónde salieron, cómo se generó el bosquejo, partición train/val/test.
* **Arquitectura:** U-Net (por qué las *skip connections*), PatchGAN 70×70 (por qué parches y no una sola decisión por imagen), función de pérdida cGAN + λ·L1.
* **Resultados:** tabla con L1 / PSNR / SSIM en test, curvas de pérdida, galería boceto · generada · real (incluye mejores **y peores** casos).
* **Discusión:** ¿qué hace bien el modelo (colores, formas globales)? ¿qué falla (detalles finos, objetos poco frecuentes, dibujos a mano distintos a Canny)? ¿cómo influye el tamaño/variedad del dataset?
* **Mejoras posibles:** más datos, *data augmentation*, entrenar más épocas, bosquejos con otro detector de bordes (HED / XDoG), probar λ distinto, comparar contra un generador entrenado solo con L1 (ablación) para mostrar el aporte de la parte adversaria.

**Referencias:** Isola et al., *Image-to-Image Translation with Conditional Adversarial Networks* (Pix2Pix, CVPR 2017) · tutorial oficial: https://www.tensorflow.org/tutorials/generative/pix2pix